## Pandas - Pactrice

In [10]:
# importar bibliotecas

import pandas as pd
import numpy as np 

# importar o dataset
df = pd.read_csv(r'../dataset/dados_streaming.csv')


# mostrar as primeiras linhas do data set
print( "First 5 rows of the dataset:")
print(df.head())
np.random.seed(42)

First 5 rows of the dataset:
   id_cliente  idade  mensalidade  meses_contrato suporte_tecnico    target
0        1000   56.0        17.60              12             Não      Fiel
1        1001   69.0        12.06              44             Não      Fiel
2        1002   46.0        16.13              36             Não  Cancelou
3        1003   32.0        10.75               4             Não  Cancelou
4        1004   60.0        10.42               5             Não      Fiel


## PHASE 1 - Data audit


In [11]:

df['idade'].isna().sum()

np.int64(15)

In [12]:
df = pd.read_csv(r'../dataset/dados_streaming.csv')

# Cleaning: How many customers are missing the age? Fill in missing age values with the median age
missing_age = df['idade'].isna().sum()
print(f"Number of customers missing age: {missing_age}")

median_age = df['idade'].median()
df.fillna({'idade': median_age}, inplace=True)

# Cardinality: How many unique values in suporte_tecnico and target?
unique_suporte_tecnico = df['suporte_tecnico'].nunique()
unique_target = df['target'].nunique()
print(f"Unique values in suport tecnico : {unique_suporte_tecnico}")
print(f"Unique values in target : {unique_target}")


# Frequency: What percentage of customers have "Canceled" the service?
canceled_percentage = df[df['target'] == 'Cancelou'].shape[0] / df.shape[0] * 100
print(f"Percentage of customers who have canceled the service: {canceled_percentage:.2f}%")



Number of customers missing age: 15
Unique values in suport tecnico : 2
Unique values in target : 2
Percentage of customers who have canceled the service: 29.00%


## PHASE 2 - Filters and Logic

In [13]:
# Segmentation: Create a new DataFrame called clientes_premium that contains only those who pay a monthly fee greater than 15.00 and have been in the service for more than 12 months.

clientes_premium = df[(df['mensalidade'] > 15.00) & (df['meses_contrato'] > 12)]
print(clientes_premium[['id_cliente', 'mensalidade', 'meses_contrato']])



     id_cliente  mensalidade  meses_contrato
2          1002        16.13              36
5          1005        15.21              37
7          1007        16.27              42
8          1008        17.16              28
9          1009        19.66              31
..          ...          ...             ...
485        1485        15.33              18
489        1489        18.98              28
493        1493        19.37              40
495        1495        15.76              32
499        1499        18.62              34

[181 rows x 3 columns]


In [14]:
# Removal: Does the id_cliente column help predict user behavior? 
# If not, remove it from the DataFrame.

df.drop(columns=['id_cliente'])




,idade,mensalidade,meses_contrato,suporte_tecnico,target
0,56.0,17.60,12,Não,Fiel
1,69.0,12.06,44,Não,Fiel
2,46.0,16.13,36,Não,Cancelou
3,32.0,10.75,4,Não,Cancelou
4,60.0,10.42,5,Não,Fiel
...,...,...,...,...,...
495,71.0,15.76,32,Sim,Cancelou
496,60.0,14.96,29,Não,Fiel
497,54.0,16.01,3,Não,Fiel
498,50.0,10.08,40,Sim,Cancelou


## PHASE 2 - Model Preparation (Encoding)

In [15]:

df = pd.read_csv(r'../dataset/dados_streaming.csv')

# Manual Mapping: Turn the target column into numbers: True -> 0, Canceled -> 1.
df['target'] = df['target'].map({'Fiel': 0, 'Cancelou': 1})
print (df)

# Binary Mapping: Transform column suporte_tecnico into: No -> 0, Yes -> 1.
df['suporte_tecnico'] = df['suporte_tecnico'].map({'Não': 0, 'Sim': 1})
print(df)


     id_cliente  idade  mensalidade  meses_contrato suporte_tecnico  target
0          1000   56.0        17.60              12             Não       0
1          1001   69.0        12.06              44             Não       0
2          1002   46.0        16.13              36             Não       1
3          1003   32.0        10.75               4             Não       1
4          1004   60.0        10.42               5             Não       0
..          ...    ...          ...             ...             ...     ...
495        1495   71.0        15.76              32             Sim       1
496        1496   60.0        14.96              29             Não       0
497        1497   54.0        16.01               3             Não       0
498        1498   50.0        10.08              40             Sim       1
499        1499   25.0        18.62              34             Não       0

[500 rows x 6 columns]
     id_cliente  idade  mensalidade  meses_contrato  suporte_tec